# Complejidad de algoritmos: medición empírica y gráfica

Ejemplos para la materia **Estructura de Datos** (1.5 Análisis de algoritmos). En vez de solo calcular la complejidad Big-O en papel, aquí la **medimos de verdad**: cada algoritmo se corre con tamaños de entrada crecientes, se cronometra con `time.perf_counter()`, se guardan los resultados en un `DataFrame` de pandas, y se grafican con matplotlib — la forma de la curva resultante debe coincidir con la complejidad teórica.

Complejidades cubiertas: **O(1)**, **O(log n)**, **O(n)**, **O(n log n)**, **O(n²)**, **O(2ⁿ)**.


In [ ]:
import time
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

random.seed(42)


def medir(func, *args, repeticiones=5):
    """Corre func varias veces y devuelve el tiempo promedio en segundos."""
    tiempos = []
    for _ in range(repeticiones):
        inicio = time.perf_counter()
        func(*args)
        tiempos.append(time.perf_counter() - inicio)
    return sum(tiempos) / len(tiempos)


## O(1) — Acceso constante

Acceder a un elemento por índice no depende del tamaño de la lista — el tiempo debe verse plano sin importar qué tan grande crezca `n`.


In [ ]:
def acceso_constante(lista):
    return lista[len(lista) // 2]


tamanos_o1 = [1_000, 10_000, 100_000, 1_000_000, 5_000_000]
resultados_o1 = []
for n in tamanos_o1:
    lista = list(range(n))
    tiempo = medir(acceso_constante, lista)
    resultados_o1.append({"n": n, "tiempo": tiempo, "algoritmo": "O(1) — acceso por índice"})

df_o1 = pd.DataFrame(resultados_o1)
df_o1


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(df_o1["n"], df_o1["tiempo"], marker="o", color="#2E7A4C")
ax.set_title("O(1) — el tiempo no crece con n")
ax.set_xlabel("Tamaño de entrada (n)")
ax.set_ylabel("Tiempo (s)")
ax.set_ylim(0, max(df_o1["tiempo"]) * 3 if max(df_o1["tiempo"]) > 0 else 1e-6)
plt.tight_layout()
plt.show()


## O(log n) — Búsqueda binaria

Cada comparación descarta la mitad de los elementos restantes. Duplicar `n` solo debería sumar **una** comparación más, no el doble de tiempo.


In [ ]:
def busqueda_binaria(lista_ordenada, objetivo):
    izq, der = 0, len(lista_ordenada) - 1
    while izq <= der:
        medio = (izq + der) // 2
        if lista_ordenada[medio] == objetivo:
            return medio
        if lista_ordenada[medio] < objetivo:
            izq = medio + 1
        else:
            der = medio - 1
    return -1


tamanos_ologn = [1_000, 10_000, 100_000, 1_000_000, 5_000_000, 10_000_000]
resultados_ologn = []
for n in tamanos_ologn:
    lista = list(range(n))
    tiempo = medir(busqueda_binaria, lista, -1)  # peor caso: no está en la lista
    resultados_ologn.append({"n": n, "tiempo": tiempo, "algoritmo": "O(log n) — búsqueda binaria"})

df_ologn = pd.DataFrame(resultados_ologn)
df_ologn


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(df_ologn["n"], df_ologn["tiempo"], marker="o", color="#3B7FAD")
ax.set_xscale("log")
ax.set_title("O(log n) — crecimiento logarítmico (nótese el eje X en escala log)")
ax.set_xlabel("Tamaño de entrada (n, escala log)")
ax.set_ylabel("Tiempo (s)")
plt.tight_layout()
plt.show()


## O(n) — Búsqueda secuencial

Sin atajos: en el peor caso hay que revisar cada elemento uno por uno. El tiempo debe crecer de forma **lineal** con `n`.


In [ ]:
def busqueda_secuencial(lista, objetivo):
    for i, valor in enumerate(lista):
        if valor == objetivo:
            return i
    return -1


tamanos_on = [10_000, 50_000, 100_000, 200_000, 400_000, 800_000]
resultados_on = []
for n in tamanos_on:
    lista = list(range(n))
    tiempo = medir(busqueda_secuencial, lista, -1)  # peor caso: no está
    resultados_on.append({"n": n, "tiempo": tiempo, "algoritmo": "O(n) — búsqueda secuencial"})

df_on = pd.DataFrame(resultados_on)
df_on


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(df_on["n"], df_on["tiempo"], marker="o", color="#A8541E")
ax.set_title("O(n) — crecimiento lineal")
ax.set_xlabel("Tamaño de entrada (n)")
ax.set_ylabel("Tiempo (s)")
plt.tight_layout()
plt.show()


## O(n log n) — Ordenamiento (Timsort de Python)

`sorted()` en Python usa Timsort, con complejidad O(n log n). Debe crecer más rápido que O(n) pero mucho más lento que O(n²).


In [ ]:
def ordenar(lista):
    return sorted(lista)


tamanos_onlogn = [10_000, 50_000, 100_000, 200_000, 400_000, 800_000]
resultados_onlogn = []
for n in tamanos_onlogn:
    lista = [random.randint(0, 10_000_000) for _ in range(n)]
    tiempo = medir(ordenar, lista, repeticiones=3)
    resultados_onlogn.append({"n": n, "tiempo": tiempo, "algoritmo": "O(n log n) — sorted()"})

df_onlogn = pd.DataFrame(resultados_onlogn)
df_onlogn


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(df_onlogn["n"], df_onlogn["tiempo"], marker="o", color="#6C5CC4")
ax.set_title("O(n log n) — ordenamiento con sorted()")
ax.set_xlabel("Tamaño de entrada (n)")
ax.set_ylabel("Tiempo (s)")
plt.tight_layout()
plt.show()


## O(n²) — Detectar duplicados con doble ciclo

Un ciclo anidado dentro de otro: por cada elemento, se recorren (casi) todos los demás. Duplicar `n` debe **cuadruplicar** el tiempo, no solo duplicarlo.


In [ ]:
def tiene_duplicados_n2(lista):
    for i in range(len(lista)):
        for j in range(i + 1, len(lista)):
            if lista[i] == lista[j]:
                return True
    return False


tamanos_on2 = [200, 400, 800, 1_600, 3_200]
resultados_on2 = []
for n in tamanos_on2:
    lista = list(range(n))  # sin duplicados -> fuerza el peor caso (recorre todo)
    tiempo = medir(tiene_duplicados_n2, lista, repeticiones=3)
    resultados_on2.append({"n": n, "tiempo": tiempo, "algoritmo": "O(n²) — doble ciclo"})

df_on2 = pd.DataFrame(resultados_on2)
df_on2


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(df_on2["n"], df_on2["tiempo"], marker="o", color="#C0392B")
ax.set_title("O(n²) — crecimiento cuadrático")
ax.set_xlabel("Tamaño de entrada (n)")
ax.set_ylabel("Tiempo (s)")
plt.tight_layout()
plt.show()


## O(2ⁿ) — Fibonacci recursivo sin memoización

El caso clásico de crecimiento exponencial: cada llamada genera dos llamadas más. Aquí `n` crece de uno en uno y el tiempo se **duplica aproximadamente en cada paso** — por eso el rango de `n` es mucho más chico que en los ejemplos anteriores.


In [ ]:
def fibonacci_recursivo(n):
    if n <= 1:
        return n
    return fibonacci_recursivo(n - 1) + fibonacci_recursivo(n - 2)


tamanos_o2n = [20, 22, 24, 26, 28, 30]
resultados_o2n = []
for n in tamanos_o2n:
    tiempo = medir(fibonacci_recursivo, n, repeticiones=1)
    resultados_o2n.append({"n": n, "tiempo": tiempo, "algoritmo": "O(2ⁿ) — fibonacci recursivo"})

df_o2n = pd.DataFrame(resultados_o2n)
df_o2n


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(df_o2n["n"], df_o2n["tiempo"], marker="o", color="#8A5A20")
ax.set_yscale("log")
ax.set_title("O(2ⁿ) — crecimiento exponencial (eje Y en escala log)")
ax.set_xlabel("Tamaño de entrada (n)")
ax.set_ylabel("Tiempo (s, escala log)")
plt.tight_layout()
plt.show()


## Comparación final: las 6 complejidades juntas

Todas normalizadas a su propio rango de `n` para poder verlas en una sola figura — el eje Y en escala logarítmica es lo que permite comparar algoritmos que van de microsegundos a segundos en la misma gráfica.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

datasets = [
    (df_o1, "O(1)", "#2E7A4C"),
    (df_ologn, "O(log n)", "#3B7FAD"),
    (df_on, "O(n)", "#A8541E"),
    (df_onlogn, "O(n log n)", "#6C5CC4"),
    (df_on2, "O(n²)", "#C0392B"),
    (df_o2n, "O(2ⁿ)", "#8A5A20"),
]

for df, etiqueta, color in datasets:
    ax.plot(df["n"], df["tiempo"], marker="o", label=etiqueta, color=color)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Tamaño de entrada (n, escala log)")
ax.set_ylabel("Tiempo (s, escala log)")
ax.set_title("Comparación de las 6 complejidades — cada una con su propio rango de n")
ax.legend()
plt.tight_layout()
plt.show()


## Tabla resumen

Todos los resultados juntos en un solo `DataFrame`, ordenado por complejidad y tamaño — la base para cualquier análisis posterior (promedio, ratio de crecimiento entre pasos, etc.).


In [ ]:
df_todos = pd.concat([df_o1, df_ologn, df_on, df_onlogn, df_on2, df_o2n], ignore_index=True)
df_todos["tiempo_ms"] = df_todos["tiempo"] * 1000
df_todos[["algoritmo", "n", "tiempo_ms"]]


## Ejercicios

1. Agrega una columna `razon_crecimiento` a cada `DataFrame` que calcule `tiempo[i] / tiempo[i-1]` — para O(n²) esa razón debe acercarse a 4 cuando `n` se duplica; para O(n log n) debe ser un poco más de 2; para O(1) debe rondar 1.
2. Implementa **merge sort** (O(n log n) "a mano", sin usar `sorted()`) y compara su curva contra la de `sorted()` — ¿cuál es más rápida en la práctica y por qué, si ambas son la misma complejidad teórica?
3. Cambia `busqueda_binaria` para que compare siempre a favor de la mitad izquierda — ¿sigue siendo O(log n)? Mide y confirma con una gráfica.
4. Agrega **memoización** a `fibonacci_recursivo` (con un diccionario o `functools.lru_cache`) y grafica el resultado en el mismo rango de `n` que la versión sin memoizar — ¿a qué complejidad salta?
